# CE 310 — Week 12 Assignment
## Sensitivity Analysis

**60 points.** Problem 1 is common to everyone; Problem 2 depends on your
declared major; Problem 3 is a short reflection.

Upload both CSV files to this Colab session before running anything.


## Setup


In [ ]:
# ── Identify your submission ─────────────────────────────────
# Fill these in before you run anything else. NETID is what matches your
# work to your student record — a blank NETID takes a 5-point deduction,
# and the track you do not declare in MAJOR is not graded at all.
MAJOR = ""   # "CE" for Track A, "ArcE" for Track B
NAME  = ""   # e.g. "Jordan Reyes"
NETID = ""   # e.g. "abc123"

print(f"NAME  = {NAME}")
print(f"NETID = {NETID}")
print(f"MAJOR = {MAJOR}")


Load the data and define the cost model.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

items    = pd.read_csv('CE310_BidItems_2024_2026.csv')
projects = pd.read_csv('CE310_BidProjects_2024_2026.csv')

def cost(q, p, oh, esc):
    """Item cost: quantity x unit price x overhead markup x escalation."""
    return q * p * (1 + oh) * (1 + esc)

print(f"line items {len(items):,}   awarded contracts {len(projects):,}")


## Problem 1 — A Different Item (30 pts)

### P1a (8 pts) — Fit and Compare the Elasticity

Fit `logp ~ logq + LOW` to **`CEM STABIL BKFL`** and report the elasticity and
the residual RMSE.


In [ ]:
cs = items[items['ItemDescription'] == 'CEM STABIL BKFL'].copy()
cs['LOW']  = (cs['LowBidder'] == 'Yes').astype(int)
cs['logq'] = np.log10(cs['Quantity'])
cs['logp'] = np.log10(cs['UnitPrice_USD'])

mb = smf.ols(___, data=cs).fit()          # fill in the model formula
RMSE_b = np.sqrt(mb.mse_resid)

print(mb.summary().tables[1])
print(f'\nn = {len(cs):,}   RMSE = {RMSE_b:.4f} log10 units   R2 = {mb.rsquared:.4f}')

ANSWER_P1a_elasticity = round(mb.params['logq'], 4)
ANSWER_P1a_rmse       = round(RMSE_b, 4)
print(f'ANSWER_P1a_elasticity = {ANSWER_P1a_elasticity}')
print(f'ANSWER_P1a_rmse = {ANSWER_P1a_rmse}')


*Written response (4 pts) — replace this text.*

Compare to excavation's −0.1660. Which item shows the stronger economy of scale,
and why? What does the RMSE difference say about predictability?


### P1b (8 pts) — The Tornado on Backfill

Same four inputs, same ranges, at Q = 500 CY.


In [ ]:
Qb = 500
Pb = 10 ** (mb.params['Intercept'] + mb.params['logq'] * np.log10(Qb) + mb.params['LOW'])
base_b = cost(Qb, Pb, 0.13, 0.05)

sw_b = {   # fill in the unit-price row: +1 and -1 RMSE, as in the in-class exercise
    'Unit price': abs(cost(Qb, Pb*10**___, .13, .05) - cost(Qb, Pb*10**___, .13, .05)),
    'Quantity':   abs(cost(Qb*1.15, Pb, .13, .05)       - cost(Qb*0.85, Pb, .13, .05)),
    'Overhead':   abs(cost(Qb, Pb, .18, .05)            - cost(Qb, Pb, .08, .05)),
    'Escalation': abs(cost(Qb, Pb, .13, .08)            - cost(Qb, Pb, .13, .02)),
}

print(f'unit price at {Qb} CY = ${Pb:.2f}   base cost = ${base_b:,.0f}\n')
for k, v in sorted(sw_b.items(), key=lambda x: -x[1]):
    print(f'  {k:<12} ${v:>10,.0f}   ({100*v/base_b:5.1f}% of base)')
print()

ANSWER_P1b_base_cost   = round(base_b, 0)
ANSWER_P1b_swing_price = round(sw_b['Unit price'], 0)
print(f'ANSWER_P1b_base_cost = {ANSWER_P1b_base_cost}')
print(f'ANSWER_P1b_swing_price = {ANSWER_P1b_swing_price}')


*Written response (4 pts) — replace this text.*

Report all four swings. Does the ranking match the exercise's? Give the price swing
as a percentage of base for both items and explain the difference via the RMSEs.


### P1c (7 pts, written) — Why the Ranking Is Stable

*Replace this text with your 4–6 sentence answer.*

What property of `Q × P × (1+OH) × (1+Esc)` makes the ranking insensitive to
both the design point and the item? What realistic change would reorder it, and
which two inputs would swap?


### P1d (7 pts) — The Survey That Would Be Worth Buying

Narrow the take-off range from ±15% to ±4% and measure what that buys.


In [ ]:
swing_before = sw_b['Quantity']
swing_after  = abs(cost(Qb*___, Pb, .13, .05) - cost(Qb*___, Pb, .13, .05))   # +/-4%

print(f'quantity swing at +/-15% = ${swing_before:,.0f}')
print(f'quantity swing at +/-4%  = ${swing_after:,.0f}')
print(f'reduction                = ${swing_before - swing_after:,.0f}'
      f'  ({100*(swing_before-swing_after)/base_b:.1f}% of base)')
print(f'unit-price swing (unchanged) = ${sw_b["Unit price"]:,.0f}')
print()

ANSWER_P1d_swing_after = round(swing_after, 0)
print(f'ANSWER_P1d_swing_after = {ANSWER_P1d_swing_after}')


*Written response (4 pts) — replace this text.*

Is the survey worth commissioning? Compare the reduction against the unit-price
swing it does nothing about, and say what you would need to know about its cost.


## Problem 2 — Track Application (18 pts)

Run **only** the track matching your declared major.

### Track A — CE: Where the Programme's Money Actually Is


In [ ]:
exc = items[items['ItemDescription'] == 'EXCAV (ROADWAY)'].copy()
exc['LOW']  = (exc['LowBidder'] == 'Yes').astype(int)
exc['logq'] = np.log10(exc['Quantity'])
exc['logp'] = np.log10(exc['UnitPrice_USD'])
me = smf.ols(___, data=exc).fit()   # fill in: the exercise's price model
RMSE_e = np.sqrt(me.mse_resid)

won = exc[exc['LowBidder'] == ___]   # fill in: awarded work only
prog_qty  = won['Quantity'].sum()
prog_cost = (won['Quantity'] * won['UnitPrice_USD']).sum()
price_exposure = prog_cost * (10**___ - 1)   # fill in: one RMSE

print(f'programme quantity {prog_qty:,.0f} CY')
print(f'programme cost     ${prog_cost:,.0f}')
print(f'price exposure     ${price_exposure:,.0f}  (+1 RMSE)')
print()

ANSWER_A2a_prog_cost = round(prog_cost, 0)
print(f'ANSWER_A2a_prog_cost = {ANSWER_A2a_prog_cost}')


**(A2-b)** Programme exposure to take-off error, at ±15% and halved.


In [ ]:
qty_exposure_15 = prog_cost * ___ * 2   # fill in: the take-off range
qty_exposure_75 = prog_cost * 0.075 * 2

print(f'quantity exposure at +/-15%  ${qty_exposure_15:,.0f}')
print(f'quantity exposure at +/-7.5% ${qty_exposure_75:,.0f}')
print()

ANSWER_A2b_qty_exposure = round(qty_exposure_15, 0)
print(f'ANSWER_A2b_qty_exposure = {ANSWER_A2b_qty_exposure}')


**(A2-c)** The comparison that decides the budget.


In [ ]:
ratio = ___ / ___   # fill in: which exposure over which

print(f'price exposure    ${price_exposure:,.0f}')
print(f'quantity exposure ${qty_exposure_15:,.0f}')
print(f'ratio = {ratio:.2f}')
print()

ANSWER_A2c_ratio = round(ratio, 2)
print(f'ANSWER_A2c_ratio = {ANSWER_A2c_ratio}')


*Track A written response (9 pts) — replace this text.*

Report both exposures and the ratio. What would spending the whole budget on
surveying buy, and what would it not? What would have to change for surveying to
be right? Why might price exposure be less reducible?


### Track B — ArcE: How Long Will It Take, and What Does Late Cost?


In [ ]:
con = projects[projects['ProjectType'] == 'Construction'].copy()
con['logd'] = np.log10(con['WorkingDays'])
con['logs'] = np.log10(con['WinningBid_USD'])

md_ = smf.ols(___, data=con).fit()      # fill in: duration on contract value
print(md_.summary().tables[1])
print(f'\nn = {len(con):,}   R2 = {md_.rsquared:.4f}')

ANSWER_B2a_slope = round(md_.params['logs'], 4)
print(f'ANSWER_B2a_slope = {ANSWER_B2a_slope}')


**(B2-b)** Predicted duration at $2.5M, and at ±25% contract value.


In [ ]:
def days(value):
    return 10 ** (md_.params['Intercept'] + md_.params['logs'] * np.log10(___))   # fill in

S0 = 2_500_000
print(f'${S0:,}      -> {days(S0):.0f} working days')
print(f'${S0*1.25:,.0f} -> {days(S0*1.25):.0f} working days')
print(f'${S0*0.75:,.0f} -> {days(S0*0.75):.0f} working days')
print()

ANSWER_B2b_days = round(days(S0), 0)
print(f'ANSWER_B2b_days = {ANSWER_B2b_days}')


**(B2-c)** What that duration swing costs at $12,000 per day.


In [ ]:
LD = 12_000
swing_days = days(S0*___) - days(S0*___)   # fill in: +/-25% on contract value
ld_cost = swing_days * LD

print(f'duration swing = {swing_days:.0f} days')
print(f'LD cost        = ${ld_cost:,.0f}  at ${LD:,}/day')
print()

ANSWER_B2c_ld_cost = round(ld_cost, 0)
print(f'ANSWER_B2c_ld_cost = {ANSWER_B2c_ld_cost}')


*Track B written response (9 pts) — replace this text.*

Report the slope, three durations and the LD figure. What does a slope below 1.0
mean for how duration scales with size, and why is that expected? Then: R² is
about 0.52 — what does that do to your confidence in the LD figure, and what
would you want before putting it in a risk register?


## Problem 3 — Written Reflection (12 pts)

*Replace this text with your 100–150 word reflection.*

One situation where the dominant input is worth spending money to pin down, and
one where it is not and you should manage the risk instead. A specific number
from this week for each.


## Before You Submit

- [ ] `NAME` and `NETID` filled in at the top and showing in the cell output — a blank `NETID` takes a 5-point deduction
- [ ] `MAJOR` set to `"CE"` or `"ArcE"` — the track you do not declare is not graded
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Every `___` replaced
- [ ] Only your declared track's cells run
- [ ] Every written response replaces its placeholder text
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `CE310_W12_Assignment_<NetID>.ipynb` and uploaded to the Week 12 Assignment folder on D2L